# 06 — Red con balance detallado y diagnóstico del cuello de botella del deuterio

Este notebook crea una segunda red `pynucastro_db`, con tasas inversas añadidas por balance detallado siempre que la instalación local de `pynucastro` lo permita. El objetivo no es esconder la discrepancia de D/H, sino probar la hipótesis física que salió de la comparación con AlterBBN:

1. `pynucastro` reproduce razonablemente $Y_p$, porque casi todos los neutrones iniciales acaban en $^4$He.
2. Sin embargo, D/H sale demasiado bajo.
3. Como $^3$He/H no está tan lejos en escala, el problema no parece ser que nunca se forme deuterio, sino que se quema demasiado eficientemente una vez formado.
4. La sospecha natural es que el cuello de botella del deuterio está incompleto: faltan o no pesan suficientemente las reacciones inversas/fotodesintegraciones, en especial $d + \gamma 
ightarrow n+p$.

El script `06_make_detailed_balance_network.py` genera la red, exporta un diagnóstico de canales inversos y, si la integración funciona, añade `pynucastro_db` a las tablas y figuras comparativas.

In [1]:
from pathlib import Path
import subprocess
import sys

script = Path('06_make_detailed_balance_network.py')
if not script.exists():
    raise FileNotFoundError(script)

# El script escribe un CSV de estado y no debe romper el flujo si falla la red DB.
result = subprocess.run([sys.executable, str(script)], text=True, capture_output=True)
print(result.stdout)
if result.stderr:
    print('STDERR:')
    print(result.stderr)
print('return code:', result.returncode)


Detailed-balance candidate network written with 76 rates.
Important reverse channels found: 7/7
Traceback (most recent call last):
  File "/home/daniel/GitHub/BNN_simulation/main/06_make_detailed_balance_network.py", line 582, in main
    df_db = integrate_db_network()
            ^^^^^^^^^^^^^^^^^^^^^^
  File "/home/daniel/GitHub/BNN_simulation/main/06_make_detailed_balance_network.py", line 458, in integrate_db_network
    raise RuntimeError(sol.message)
RuntimeError: Required step size is less than spacing between numbers.


return code: 0


In [2]:
import pandas as pd
from pathlib import Path

def show_csv(path):
    path = Path(path)
    print(f"\n--- {path} ---")
    if path.exists():
        display(pd.read_csv(path))
    else:
        print("No existe todavía")

show_csv('data/pynucastro_db_status.csv')
show_csv('data/bbn_network_reverse_diagnostic.csv')
show_csv('data/bbn_network_db_derived_attempts.csv')
show_csv('data/bbn_final_abundances_pynucastro_db.csv')


--- data/pynucastro_db_status.csv ---


,status,message,n_rates,reverse_channels_found,reverse_channels_total,traceback
0,integration_failed,"Network was built, but integration failed: Run...",76,7,7,"Traceback (most recent call last):\n File ""/h..."



--- data/bbn_network_reverse_diagnostic.csv ---


,channel,reactants_expected,products_expected,found,matched_rate,Q_MeV,rate_class,derived_from_inverse,note,diagnostic
0,d_to_n_p,d,n + p,True,H2 ⟶ n + p,-2.224566,DerivedRate,True,"photodisintegration counterpart of p(n,gamma)d",present in bbn_network_db.py
1,he3_to_p_d,he3,p + d,True,He3 ⟶ p + H2,-5.493475,DerivedRate,True,"reverse of d(p,gamma)he3",present in bbn_network_db.py
2,t_to_n_d,t,n + d,True,H3 ⟶ n + H2,-6.257230,DerivedRate,True,"reverse of d(n,gamma)t",present in bbn_network_db.py
3,he4_to_p_t,he4,p + t,True,He4 ⟶ p + H3,-19.813866,DerivedRate,True,"reverse of t(p,gamma)he4",present in bbn_network_db.py
4,he4_to_n_he3,he4,n + he3,True,He4 ⟶ n + He3,-20.577621,DerivedRate,True,"reverse of he3(n,gamma)he4",present in bbn_network_db.py
5,be7_to_he4_he3,be7,he4 + he3,True,Be7 ⟶ He4 + He3,-1.587135,DerivedRate,True,"reverse of he3(alpha,gamma)be7",present in bbn_network_db.py
6,li7_to_he4_t,li7,he4 + t,True,Li7 ⟶ He4 + H3,-2.467622,DerivedRate,True,"reverse of t(alpha,gamma)li7",present in bbn_network_db.py



--- data/bbn_network_db_derived_attempts.csv ---


,source_rate,derived_rate,derived_ok,message,source_reactants,source_products,source_Q_MeV
0,n + p --> d <reaclib_an06>,H2 ⟶ n + p,True,created by Library.derived_backward(use_pf=False),NaN,NaN,NaN
1,d + n --> t <reaclib_nk06>,H3 ⟶ n + H2,True,created by Library.derived_backward(use_pf=False),NaN,NaN,NaN
2,d + p --> He3 <reaclib_de04>,He3 ⟶ p + H2,True,created by Library.derived_backward(use_pf=False),NaN,NaN,NaN
3,d + d --> He4 <reaclib_nacr>,He4 ⟶ H2 + H2,True,created by Library.derived_backward(use_pf=False),NaN,NaN,NaN
4,d + He4 --> Li6 <reaclib_tu19>,Li6 ⟶ He4 + H2,True,created by Library.derived_backward(use_pf=False),NaN,NaN,NaN
...,...,...,...,...,...,...,...
72,Li7 + He3 ⟶ n + p + He4 + He4,n + p + He4 + He4 ⟶ He3 + Li7,True,created by explicit DerivedRate(source_rate=.....,He3 + Li7,n + p + He4 + He4,9.62776
73,Be7 + H3 ⟶ n + p + He4 + He4,n + p + He4 + He4 ⟶ H3 + Be7,True,created by explicit DerivedRate(source_rate=.....,t + Be7,n + p + He4 + He4,10.50880
74,Be7 + He3 ⟶ p + p + He4 + He4,p + p + He4 + He4 ⟶ He3 + Be7,True,created by explicit DerivedRate(source_rate=.....,He3 + Be7,p + p + He4 + He4,11.27210
75,n + p + He4 ⟶ Li6 + 𝛾,Li6 ⟶ n + p + He4,True,created by explicit DerivedRate(source_rate=.....,n + p + He4,Li6,3.70000



--- data/bbn_final_abundances_pynucastro_db.csv ---
No existe todavía


## Qué mirar

La prueba decisiva es la fila `d_to_n_p` en `bbn_network_reverse_diagnostic.csv`. Si aparece como `found = True`, la red `pynucastro_db` sí contiene una vía inversa directa para la reacción de formación del deuterio. Después hay que mirar el valor final de D/H en `pynucastro_db`:

- Si D/H sube hacia $10^{-5}$, la discrepancia principal venía de no tratar bien el equilibrio directo-inverso del deuterio.
- Si D/H sigue muy bajo, entonces la limitación no es solo la red nuclear: también pesa la trayectoria termodinámica impuesta, la ausencia de evolución cosmológica completa, las tasas débiles completas y la física de fotones, electrones, positrones y neutrinos.

El notebook `05_alterbbn_benchmark.ipynb` detecta automáticamente la fila `pynucastro_db` si existe en `bbn_final_abundances_all_models.csv`. Por tanto, después de este notebook conviene ejecutar de nuevo el benchmark para regenerar las figuras finales.